In [16]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================

import re
import pandas as pd
import numpy as np

from tqdm import tqdm

pd.set_option("display.max_columns", None)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [17]:
# ============================================================
# CELL 2 — LOAD DATASET
# ============================================================

DATASET_PATH = "../parquet_exports/gold_ticket_similarity.parquet"

df = pd.read_parquet(DATASET_PATH)

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print(f"Rows    : {len(df):,}")
print(f"Columns : {df.shape[1]}")

df.head()

DATASET LOADED
Rows    : 230,114
Columns : 29


,similarity_pk,ticket_pk,source_system,ticket_id,created_at,closed_at,text_corpus,synthetic_text_corpus,similarity_method,embedding_strategy,text_source_type,embedding_model,priority_encoded,urgency_encoded,impact_encoded,followup_count_bucket,corpus_quality_score,similarity_confidence,followup_count,avg_followup_content_length,private_followup_ratio,url_content_ratio,meaningful_content_ratio,has_os_context,has_hardware_context,has_bios_context,has_software_context,similar_ticket_ids,similarity_scores
0,4ceef0c770bdc93599fb1d7131032c69,2013_2,GLPI,2,2013-05-13 17:27:43,2013-05-16 12:29:40,NaN,prio_high urg_critical impact_medium ticket_un...,synthetic_context,structured_metadata,synthetic_context,NaN,4.0,5.0,3.0,2.0,0.3,0.25,3,271.00,0.0,0.0,1.0,0,0,0,0,None,None
1,6cc7ce0f8b7df1ecd0560860283acf40,2013_3,GLPI,3,2013-05-14 17:38:55,2013-06-30 10:55:47,NaN,prio_medium urg_medium impact_medium ticket_un...,synthetic_context,structured_metadata,synthetic_context,NaN,3.0,3.0,3.0,2.0,0.3,0.25,3,127.00,0.0,0.0,1.0,0,0,0,0,None,None
2,23142610733bcac6466bce80d180c346,2013_4,GLPI,4,2013-05-15 13:01:00,2013-07-01 19:46:26,NaN,prio_medium urg_medium impact_medium ticket_un...,synthetic_context,structured_metadata,synthetic_context,NaN,3.0,3.0,3.0,5.0,0.5,0.45,78,130.38,0.0,0.0,1.0,0,0,0,0,None,None
3,10ac3171da0e097808f00c3ad5fb994d,2013_5,GLPI,5,2013-05-15 13:21:00,2013-06-20 19:25:32,NaN,prio_critical urg_critical impact_high ticket_...,synthetic_context,structured_metadata,synthetic_context,NaN,5.0,5.0,4.0,2.0,0.3,0.25,3,258.00,0.0,0.0,1.0,0,0,0,0,None,None
4,49e2a88ee2717c5a044429d4da872925,2013_6,GLPI,6,2013-05-15 14:44:54,2013-06-30 10:55:47,NaN,prio_medium urg_medium impact_medium ticket_un...,synthetic_context,structured_metadata,synthetic_context,NaN,3.0,3.0,3.0,1.0,0.3,0.25,0,0.00,0.0,0.0,0.0,0,0,0,0,None,None


In [18]:
# ============================================================
# CELL 3 — BUILD RETRIEVAL TEXT
# ============================================================

work_df = df.copy()

print("=" * 60)
print("TEXT SOURCE DISTRIBUTION")
print("=" * 60)

print(work_df["text_source_type"].value_counts())

# ------------------------------------------------------------
# KEEP ONLY REAL TICKET TEXT
# ------------------------------------------------------------

before_rows = len(work_df)

work_df = work_df[
    work_df["text_source_type"] == "real_text"
].copy()

after_rows = len(work_df)

print("\nRows before filter :", f"{before_rows:,}")
print("Rows after filter  :", f"{after_rows:,}")
print("Removed synthetic  :", f"{before_rows - after_rows:,}")

# ------------------------------------------------------------
# BUILD RETRIEVAL TEXT
# ------------------------------------------------------------

work_df["retrieval_text"] = (
    work_df["text_corpus"]
    .fillna("")
    .astype(str)
)

# Remove empty texts
work_df = work_df[
    work_df["retrieval_text"].str.strip() != ""
].copy()

print("\nRetrieval text created successfully.")

print("\nSample retrieval text:")
print(work_df["retrieval_text"].iloc[0][:500])

TEXT SOURCE DISTRIBUTION
text_source_type
real_text            228587
synthetic_context      1527
Name: count, dtype: int64

Rows before filter : 230,114
Rows after filter  : 228,587
Removed synthetic  : 1,527

Retrieval text created successfully.

Sample retrieval text:
The payment was deducted from my bank account but the transaction shows failed. The payment was deducted from my bank account but the transaction shows failed.


In [19]:
print("=" * 60)
print("TEXT VALIDATION")
print("=" * 60)

before_rows = len(work_df)

work_df = work_df[
    work_df["retrieval_text"].notna()
].copy()

work_df = work_df[
    work_df["retrieval_text"].str.len() >= 20
].copy()

print(f"Removed rows: {before_rows - len(work_df):,}")
print(f"Remaining   : {len(work_df):,}")

TEXT VALIDATION
Removed rows: 4
Remaining   : 228,583


In [20]:
# ============================================================
# CELL 4 — BASIC TEXT CLEANING FUNCTION
# ============================================================

def clean_text(text: str) -> str:
    """
    Production-oriented text cleaner
    for ticket retrieval systems.
    """

    if pd.isna(text):
        return ""

    text = str(text)

    # lowercase
    text = text.lower()

    # remove html
    text = re.sub(r"<.*?>", " ", text)

    # remove urls
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # remove emails
    text = re.sub(r"\S+@\S+", " ", text)

    # remove ip addresses
    text = re.sub(
        r"\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b",
        " ",
        text
    )

    # remove ticket ids
    text = re.sub(
        r"(ticket|inc|req)[-_ ]?\d+",
        " ",
        text
    )

    # remove repeated punctuation
    text = re.sub(r"([!?.,])\1+", r"\1", text)

    # remove extra symbols
    text = re.sub(r"[^a-zA-Z0-9\s\-_./]", " ", text)

    # remove line breaks
    text = re.sub(r"[\n\r\t]", " ", text)

    # remove multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


print("clean_text function created.")

clean_text function created.


In [21]:
# ============================================================
# CELL 5 — REMOVE COMMON TICKET NOISE
# ============================================================

NOISE_PATTERNS = [
    r"best regards",
    r"kind regards",
    r"thanks regards",
    r"thank you",
    r"sent from my iphone",
    r"please help",
    r"dear support",
    r"hello team",
    r"hi team",
    r"good morning",
    r"good afternoon",
]

def remove_ticket_noise(text: str) -> str:

    if pd.isna(text):
        return ""

    text = str(text)

    for pattern in NOISE_PATTERNS:
        text = re.sub(pattern, " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()


print("Noise removal function created.")

Noise removal function created.


In [22]:
# ============================================================
# CELL 6 — REMOVE REPEATED PHRASES
# ============================================================

def remove_repeated_phrases(text: str) -> str:

    if pd.isna(text):
        return ""

    text = str(text)

    # remove duplicated consecutive phrases
    text = re.sub(
        r"\b(.+?)(?:\s+\1\b)+",
        r"\1",
        text
    )

    return text.strip()


print("Repeated phrase cleaner created.")

Repeated phrase cleaner created.


In [23]:
# ============================================================
# CELL 7 — APPLY CLEANING PIPELINE
# ============================================================

tqdm.pandas(desc="Cleaning tickets")

work_df["retrieval_text_clean"] = (
    work_df["retrieval_text"]
    .progress_apply(clean_text)
    .progress_apply(remove_ticket_noise)
    .progress_apply(remove_repeated_phrases)
)

print("Cleaning pipeline applied successfully.")

work_df[
    [
        "retrieval_text",
        "retrieval_text_clean"
    ]
].head()


Cleaning tickets:   0%|          | 0/228583 [00:00<?, ?it/s]

Cleaning tickets: 100%|██████████| 228583/228583 [03:18<00:00, 1151.10it/s]

Cleaning pipeline applied successfully.


,retrieval_text,retrieval_text_clean
1527,The payment was deducted from my bank account ...,the payment was deducted from my bank account ...
1528,I found a bug in the latest update affecting r...,i found a bug in the latest update affecting r...
1529,The application crashes whenever I try to uplo...,the application crashes whenever i try to uplo...
1530,My subscription was cancelled without my reque...,my subscription was cancelled without my reque...
1531,The system is not syncing data across devices ...,the system is not syncing data across devices ...


In [24]:
before_rows = len(work_df)

work_df = work_df[
    work_df["retrieval_text_clean"].notna()
].copy()

work_df = work_df[
    work_df["retrieval_text_clean"].str.strip() != ""
].copy()

work_df = work_df[
    work_df["retrieval_text_clean"].str.len() >= 20
].copy()

removed_rows = before_rows - len(work_df)

print("=" * 60)
print("EMPTY / SHORT TEXT FILTER")
print("=" * 60)

print(f"Removed rows : {removed_rows:,}")
print(f"Remaining    : {len(work_df):,}")

EMPTY / SHORT TEXT FILTER
Removed rows : 0
Remaining    : 228,583


In [25]:
# ============================================================
# CELL 9 — TEXT LENGTH FEATURES
# ============================================================

work_df["text_word_count"] = (
    work_df["retrieval_text_clean"]
    .str.split()
    .apply(len)
)

work_df["text_char_count"] = (
    work_df["retrieval_text_clean"]
    .str.len()
)

print("Text length features created.")

work_df[
    [
        "text_word_count",
        "text_char_count"
    ]
].describe()

Text length features created.


,text_word_count,text_char_count
count,228583.000000,228583.000000
mean,26.836151,168.956978
std,16.646985,119.307824
min,2.000000,23.000000
25%,20.000000,113.000000
50%,22.000000,137.000000
75%,26.000000,149.000000
max,201.000000,1512.000000


In [26]:
# ============================================================
# CELL 10 — REMOVE VERY SHORT TICKETS
# ============================================================

MIN_WORDS = 5

before_filter = len(work_df)

work_df = work_df[
    work_df["text_word_count"] >= MIN_WORDS
].copy()

removed_short = before_filter - len(work_df)

print("=" * 60)
print("SHORT TICKET FILTER")
print("=" * 60)

print(f"Removed short tickets : {removed_short:,}")
print(f"Remaining tickets     : {len(work_df):,}")

SHORT TICKET FILTER
Removed short tickets : 22
Remaining tickets     : 228,561


In [27]:
# ============================================================
# CELL 11 — REMOVE DUPLICATED TICKETS
# ============================================================

before_duplicates = len(work_df)

if "ticket_pk" in work_df.columns:

    work_df = (
        work_df
        .drop_duplicates(subset=["ticket_pk"])
        .reset_index(drop=True)
    )

removed_duplicates = before_duplicates - len(work_df)

print("=" * 60)
print("DUPLICATE REMOVAL")
print("=" * 60)

print(f"Removed duplicates : {removed_duplicates:,}")
print(f"Remaining tickets  : {len(work_df):,}")

DUPLICATE REMOVAL
Removed duplicates : 0
Remaining tickets  : 228,561


In [28]:
# ============================================================
# CELL 12 — REPETITION RATIO ANALYSIS
# ============================================================

def repetition_ratio(text: str) -> float:

    words = str(text).split()

    if len(words) == 0:
        return 0.0

    return 1 - (len(set(words)) / len(words))


work_df["repetition_ratio"] = (
    work_df["retrieval_text_clean"]
    .apply(repetition_ratio)
)

print("Repetition ratio created.")

work_df["repetition_ratio"].describe()

Repetition ratio created.


count    228561.000000
mean          0.466904
std           0.114311
min           0.000000
25%           0.500000
50%           0.500000
75%           0.500000
max           0.545455
Name: repetition_ratio, dtype: float64

In [29]:
# ============================================================
# CELL 13 — REMOVE HIGHLY REPETITIVE TICKETS
# ============================================================

MAX_REPETITION_RATIO = 0.60

before_repetition = len(work_df)

work_df = work_df[
    work_df["repetition_ratio"] <= MAX_REPETITION_RATIO
].copy()

removed_repetition = before_repetition - len(work_df)

print("=" * 60)
print("REPETITION FILTER")
print("=" * 60)

print(f"Removed repetitive tickets : {removed_repetition:,}")
print(f"Remaining tickets          : {len(work_df):,}")

REPETITION FILTER
Removed repetitive tickets : 0
Remaining tickets          : 228,561


In [30]:
# ============================================================
# CELL 14 — FINAL RETRIEVAL DATAFRAME
# ============================================================

REQUIRED_COLUMNS = [
    "similarity_pk",
    "ticket_pk",
    "source_system",
    "text_source_type",
    "similarity_method",
    "retrieval_text_clean",
    "corpus_quality_score",
    "similarity_confidence",
    "priority_encoded",
    "urgency_encoded",
    "impact_encoded",
    "text_word_count",
    "text_char_count",
    "repetition_ratio"
]

existing_columns = [
    c for c in REQUIRED_COLUMNS
    if c in work_df.columns
]

missing_columns = [
    c for c in REQUIRED_COLUMNS
    if c not in work_df.columns
]

print("=" * 60)
print("COLUMN VALIDATION")
print("=" * 60)

print("Existing columns:")
print(existing_columns)

print("\nMissing columns:")
print(missing_columns)

retrieval_df = (
    work_df[existing_columns]
    .reset_index(drop=True)
)

print("=" * 60)
print("FINAL RETRIEVAL DATAFRAME")
print("=" * 60)

print(f"Rows    : {len(retrieval_df):,}")
print(f"Columns : {retrieval_df.shape[1]}")

retrieval_df.head()

COLUMN VALIDATION
Existing columns:
['similarity_pk', 'ticket_pk', 'source_system', 'text_source_type', 'similarity_method', 'retrieval_text_clean', 'corpus_quality_score', 'similarity_confidence', 'priority_encoded', 'urgency_encoded', 'impact_encoded', 'text_word_count', 'text_char_count', 'repetition_ratio']

Missing columns:
[]
FINAL RETRIEVAL DATAFRAME
Rows    : 228,561
Columns : 14


,similarity_pk,ticket_pk,source_system,text_source_type,similarity_method,retrieval_text_clean,corpus_quality_score,similarity_confidence,priority_encoded,urgency_encoded,impact_encoded,text_word_count,text_char_count,repetition_ratio
0,abf40a28ecf0e6c557ae214fb08e74ca,18ecc0b10c975e6a301035cfacf70a70,customer_support_tickets_200k,real_text,nlp_embedding,the payment was deducted from my bank account ...,1.0,0.9,NaN,NaN,NaN,26,159,0.538462
1,bb988e6a04ac835153c4b6cf016f29a8,c00c291d337d61a00615ab907353bc89,customer_support_tickets_200k,real_text,nlp_embedding,i found a bug in the latest update affecting r...,1.0,0.9,NaN,NaN,NaN,22,127,0.500000
2,d111e697dee69897b9c32be8e0832860,b2e3cefec6c77072e59e9aca6a525670,customer_support_tickets_200k,real_text,nlp_embedding,the application crashes whenever i try to uplo...,1.0,0.9,NaN,NaN,NaN,20,113,0.500000
3,62e2c2f208996587aa73c7b6c2756595,9d0370f2cab75492f184db82f51c103f,customer_support_tickets_200k,real_text,nlp_embedding,my subscription was cancelled without my reque...,1.0,0.9,NaN,NaN,NaN,22,149,0.545455
4,231e0b16e61ae075aed6bcb6e76958f7,ff258a55725d78aa9fac5bdeb364d48b,customer_support_tickets_200k,real_text,nlp_embedding,the system is not syncing data across devices ...,1.0,0.9,NaN,NaN,NaN,18,111,0.500000


In [31]:
# ============================================================
# CELL 15 — FINAL DATASET HEALTH CHECK
# ============================================================

print("=" * 60)
print("FINAL DATASET HEALTH CHECK")
print("=" * 60)

print(f"Final Rows               : {len(retrieval_df):,}")
print(f"Average Word Count       : {retrieval_df['text_word_count'].mean():.2f}")
print(f"Average Character Count  : {retrieval_df['text_char_count'].mean():.2f}")
print(f"Average Repetition Ratio : {retrieval_df['repetition_ratio'].mean():.4f}")

if "source_system" in retrieval_df.columns:

    print("\nSource Systems:")
    print(
        retrieval_df["source_system"]
        .value_counts()
    )

FINAL DATASET HEALTH CHECK
Final Rows               : 228,561
Average Word Count       : 26.84
Average Character Count  : 168.97
Average Repetition Ratio : 0.4669

Source Systems:
source_system
customer_support_tickets_200k    200000
dataset_tickets_multi_lang        28561
Name: count, dtype: int64


In [32]:
# ============================================================
# CELL 16 — EXPORT CLEAN RETRIEVAL DATASET
# ============================================================

EXPORT_PATH = "../parquet_exports/retrieval_dataset_clean.parquet"

retrieval_df.to_parquet(
    EXPORT_PATH,
    index=False
)

print("=" * 60)
print("EXPORT COMPLETED")
print("=" * 60)

print(f"Exported dataset path:\n{EXPORT_PATH}")

EXPORT COMPLETED
Exported dataset path:
../parquet_exports/retrieval_dataset_clean.parquet


In [33]:
# ============================================================
# CELL 17 — EXPORT SAMPLE CSV
# ============================================================

SAMPLE_EXPORT = "../evaluation/retrieval_dataset_sample.csv"

retrieval_df.head(100).to_csv(
    SAMPLE_EXPORT,
    index=False
)

print(f"Sample exported to:\n{SAMPLE_EXPORT}")

Sample exported to:
../evaluation/retrieval_dataset_sample.csv
